In [ ]:
!pip install kaggle ultralytics opencv-python pandas matplotlib

In [ ]:
!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign

Dataset URL: https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign
License(s): CC0-1.0
100% 612M/612M [00:07<00:00, 90.0MB/s]



In [ ]:
!unzip gtsrb-german-traffic-sign.zip

In [ ]:
import os

train_path = "/content/Train"

print(os.listdir(train_path)[:5])

['23', '28', '4', '29', '0']


In [ ]:
import pandas as pd

df = pd.read_csv("/content/Test.csv")
df.head()

,Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2,ClassId,Path
0,53,54,6,5,48,49,16,Test/00000.png
1,42,45,5,5,36,40,1,Test/00001.png
2,48,52,6,6,43,47,38,Test/00002.png
3,27,29,5,5,22,24,33,Test/00003.png
4,60,57,5,5,55,52,11,Test/00004.png


In [ ]:
import os
import cv2
import pandas as pd

base_path = "/content"
output_path = "/content/dataset"

os.makedirs(f"{output_path}/images/train", exist_ok=True)
os.makedirs(f"{output_path}/images/val", exist_ok=True)
os.makedirs(f"{output_path}/labels/train", exist_ok=True)
os.makedirs(f"{output_path}/labels/val", exist_ok=True)

df = pd.read_csv("/content/Test.csv")

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2)

def convert(df_split, split):
    for _, row in df_split.iterrows():
        img_path = os.path.join(base_path, row['Path'])

        if not os.path.exists(img_path):
            continue

        img = cv2.imread(img_path)
        h, w, _ = img.shape

        x_center = ((row['Roi.X1'] + row['Roi.X2']) / 2) / w
        y_center = ((row['Roi.Y1'] + row['Roi.Y2']) / 2) / h
        width = (row['Roi.X2'] - row['Roi.X1']) / w
        height = (row['Roi.Y2'] - row['Roi.Y1']) / h

        img_name = os.path.basename(img_path)

        cv2.imwrite(f"{output_path}/images/{split}/{img_name}", img)

        with open(f"{output_path}/labels/{split}/{img_name.replace('.png','.txt')}", "w") as f:
            f.write(f"{int(row['ClassId'])} {x_center} {y_center} {width} {height}")

convert(train_df, "train")
convert(val_df, "val")

print("✅ Converted successfully!")

✅ Converted successfully!


In [ ]:
yaml_content = """path: /content/dataset
train: images/train
val: images/val

nc: 43
names: ['0','1','2','3','4','5','6','7','8','9','10','11','12','13','14','15','16','17','18','19',
        '20','21','22','23','24','25','26','27','28','29','30','31','32','33','34','35',
        '36','37','38','39','40','41','42']
"""

with open("/content/traffic.yaml", "w") as f:
    f.write(yaml_content)

print("✅ YAML fixed!")

✅ YAML fixed!


In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov5s.pt")

model.train(
    data="/content/traffic.yaml",
    epochs=30,
    imgsz=416,
    batch=16
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PRO TIP 💡 Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.

Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/traffic.yaml, degrees=0.0, deterministic=True, device=No

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78dc2d43a330>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.03

In [ ]:
model = YOLO("runs/detect/train/weights/best.pt")

model("/content/dataset/images/val", save=True)